# DS Project: Complete Research Workflow Demo

This notebook demonstrates the **complete 7-step research workflow** of the DS Project:

**STEP 0**: Vanilla Baseline (Pure LogisticRegression + BERT)
- No hierarchy, no DS reasoning
- Baseline performance metrics

**STEP 1**: Belief Extraction (DS system baseline, no thresholds)
- Extract belief values for all intents
- Prepare data for threshold optimization

**STEP 2**: Threshold Computation
- Analyze belief distributions
- Find optimal threshold per intent

**STEP 3**: DS Evaluation WITHOUT Thresholds
- DS baseline with frequent clarifications
- Measure performance and clarification rate

**STEP 4**: DS Evaluation WITH Thresholds
- Optimized DS with selective clarifications
- Improved performance metrics

**STEP 5**: Select User Study Queries
- Identify challenging queries for human validation
- Prioritize based on clarifications and confidence

**STEP 6**: Real User Study (Manual)
- Human participants answer clarifications
- Measure real-world performance

**STEP 7**: Comparison Report
- Compare all approaches
- Show improvements and trade-offs

---

**Demo Mode**: This notebook uses small sample sizes for quick testing. Set `QUICK_DEMO = False` for full evaluation.

## Setup and Configuration

Initialize paths, load dependencies, and configure the workflow.

In [1]:
import sys
from pathlib import Path
import os

# Add project root to path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import logging
from datasets import load_dataset
import pandas as pd
import json

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
QUICK_DEMO = False  # Set to False for full evaluation

DATASET = "banking77"
BASE_DIR = project_root
EXPERIMENTS_DIR = BASE_DIR / "experiments" / DATASET
RESULTS_DIR = BASE_DIR / "results" / DATASET / "workflow_demo"
CONFIG_DIR = BASE_DIR / "config"
USER_STUDY_DIR = BASE_DIR / "outputs" / "user_study" / "workflow_demo"

# Sample sizes
if QUICK_DEMO:
    TRAIN_SAMPLES = None  # Use full training set even in demo
    EVAL_SAMPLES = 50     # Quick evaluation
    LLM_SAMPLES = 100     # Quick LLM simulation
    HUMAN_STUDY_SAMPLES = 30  # Queries for human study
else:
    TRAIN_SAMPLES = None
    EVAL_SAMPLES = None   # Full test set
    LLM_SAMPLES = None    # All 3080 queries
    HUMAN_STUDY_SAMPLES = 100

# Create directories
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
USER_STUDY_DIR.mkdir(parents=True, exist_ok=True)

print(f"   Configuration complete")
print(f"   Mode: {'QUICK DEMO' if QUICK_DEMO else 'FULL EVALUATION'}")
print(f"   Dataset: {DATASET}")
print(f"   Evaluation samples: {EVAL_SAMPLES if EVAL_SAMPLES else 'All (3080)'}")
print(f"   LLM simulation samples: {LLM_SAMPLES if LLM_SAMPLES else 'All (3080)'}")
print(f"   Human study queries: {HUMAN_STUDY_SAMPLES}")

   Configuration complete
   Mode: FULL EVALUATION
   Dataset: banking77
   Evaluation samples: All (3080)
   LLM simulation samples: All (3080)
   Human study queries: 100


## STEP 0: Vanilla Baseline (Pure LogisticRegression + BERT)

Evaluate pure Logistic Regression + BERT (no hierarchy, no DS reasoning).

**What it does:**
- Trains classifier on Banking77
- Evaluates on test set
- Computes: Accuracy, F1, Precision, Recall, Confidence stats

**Output:** `vanilla_metrics.json`, `vanilla_predictions.csv`

In [2]:
# Force reload modules to get latest code changes
import sys
import importlib
if 'src.models.embeddings' in sys.modules:
    importlib.reload(sys.modules['src.models.embeddings'])
if 'scripts.training.train' in sys.modules:
    importlib.reload(sys.modules['scripts.training.train'])

from scripts.training.train import train_model
from scripts.evaluation.evaluate_vanilla_baseline import evaluate_vanilla_baseline
import json

print("="*60)
print("STEP 0: VANILLA BASELINE EVALUATION")
print("="*60)

# Step 0.1: Train model
print("\n[0.1] Training vanilla baseline model...")
train_results = train_model(
    dataset=DATASET,
    classifier_type='logistic',
    embedding_model='intfloat/e5-base',
    output_dir=EXPERIMENTS_DIR,
    experiment_name=f"{DATASET}_logistic",
    hierarchy_file=None,
    intents_file=None,
    max_iter=1000,
    batch_size=64,
    train_samples=TRAIN_SAMPLES,
    eval_samples=None
)

# Extract objects from training results
classifier = train_results['classifier']
embedder = train_results['embedder']
data_loader = train_results['data_loader']

# Step 0.2: Evaluate on test set
print("\n[0.2] Evaluating on test set...")
vanilla_output_dir = RESULTS_DIR / "vanilla"
vanilla_output_dir.mkdir(parents=True, exist_ok=True)

metrics = evaluate_vanilla_baseline(
    classifier=classifier,
    embedder=embedder,
    intent_names=data_loader.intent_names,
    dataset_name=DATASET,
    num_samples=EVAL_SAMPLES,
    batch_size=64,
    output_dir=vanilla_output_dir
)

print("\n✅ STEP 0 Complete!")
print(f"   Accuracy: {metrics['accuracy']:.4f}")
print(f"   F1 (macro): {metrics['f1_macro']:.4f}")
print(f"   Saved to: {vanilla_output_dir}")

2026-02-05 19:33:48,037 - INFO - Starting training for banking77
2026-02-05 19:33:48,037 - INFO - Classifier: logistic
2026-02-05 19:33:48,037 - INFO - Embedding model: intfloat/e5-base
2026-02-05 19:33:48,038 - INFO - Experiment name: banking77_logistic
2026-02-05 19:33:48,038 - INFO - Loading dataset...
2026-02-05 19:33:48,038 - INFO - Loading dataset: banking77


STEP 0: VANILLA BASELINE EVALUATION

[0.1] Training vanilla baseline model...


2026-02-05 19:33:50,435 - INFO - Loaded banking77: 77 intents
2026-02-05 19:33:50,484 - INFO - Training samples: 10003
2026-02-05 19:33:50,484 - INFO - No hierarchy provided. Training vanilla baseline with intent names (STRING labels)...
2026-02-05 19:33:50,484 - INFO - Initializing sentence embedder...
2026-02-05 19:33:50,501 - INFO - Use pytorch device_name: cuda
2026-02-05 19:33:50,502 - INFO - Load pretrained SentenceTransformer: intfloat/e5-base
2026-02-05 19:33:52,215 - INFO - Loaded embedding model: intfloat/e5-base
2026-02-05 19:33:52,215 - INFO - Generating embeddings for training data...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-02-05 19:33:53,829 - INFO - Skipping intent embeddings (vanilla baseline)
2026-02-05 19:33:53,829 - INFO - Training classifier...
2026-02-05 19:33:53,830 - INFO - Initialized logistic classifier
2026-02-05 19:33:53,830 - INFO - Training logistic classifier on 10003 samples
2026-02-05 19:33:53,830 - INFO - Training with label type: <class 'str'>
2026-02-05 19:34:07,051 - INFO - Training completed
2026-02-05 19:34:07,052 - INFO - Saving model to /home/kudzai/projects/DS_Project/experiments/banking77/banking77_logistic_model.pkl
2026-02-05 19:34:07,055 - INFO - Model saved to /home/kudzai/projects/DS_Project/experiments/banking77/banking77_logistic_model.pkl
2026-02-05 19:34:07,058 - INFO - Saving embedder to /home/kudzai/projects/DS_Project/experiments/banking77/banking77_embedder.json
2026-02-05 19:34:07,058 - INFO - Saved embedder config to /home/kudzai/projects/DS_Project/experiments/banking77/banking77_embedder.json
2026-02-05 19:34:07,058 - INFO - Evaluating on test set...
2026


[0.2] Evaluating on test set...


2026-02-05 19:34:09,452 - INFO - Evaluating on 3080 test samples...
2026-02-05 19:34:09,500 - INFO - Encoding test texts...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

2026-02-05 19:34:09,929 - INFO - Making predictions...
2026-02-05 19:34:09,942 - INFO - Predictions type: <class 'numpy.str_'>, sample: card_linking
2026-02-05 19:34:09,942 - INFO - Test labels type: <class 'str'>, sample: card_arrival
2026-02-05 19:34:09,956 - INFO - Saved predictions to /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/vanilla/banking77_vanilla_predictions.csv



✅ STEP 0 Complete!
   Accuracy: 0.8766
   F1 (macro): 0.8728
   Saved to: /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/vanilla


## STEP 1: Belief Extraction (DS Baseline, No Thresholds)

Extract belief values from DS system without threshold optimization.

**What it does:**
- For each query: compute_mass_function() → compute_belief()
- Extract beliefs for all 96 nodes (intents + ancestors)
- No clarifications triggered yet

**Output:** `beliefs.csv`

In [3]:
# STEP 1: Belief Extraction
import importlib
import scripts.ds_preparation.extract_beliefs
importlib.reload(scripts.ds_preparation.extract_beliefs)
from scripts.ds_preparation.extract_beliefs import extract_beliefs_from_data

from src.models.embeddings import IntentEmbeddings
from src.models.ds_mass_function import DSMassFunction
from config.hierarchy_loader import load_hierarchy_from_json, load_hierarchical_intents_from_json
import pandas as pd

print("="*60)
print("STEP 1: BELIEF EXTRACTION")
print("="*60)

# Load configuration files
config_file = CONFIG_DIR / "hierarchies" / f"{DATASET}_hierarchy.json"
intents_file = CONFIG_DIR / "hierarchies" / f"{DATASET}_intents.json"

print("\nLoading hierarchy and intents...")
hierarchy = load_hierarchy_from_json(str(config_file))
hierarchical_intents = load_hierarchical_intents_from_json(str(intents_file))

# Create intent embeddings
print("Creating intent embeddings...")
intent_embeddings = IntentEmbeddings(hierarchical_intents, embedder=embedder)

# Initialize DS Mass Function (WITHOUT thresholds for belief extraction)
print("Initializing DS Mass Function...")
ds_calculator = DSMassFunction(
    intent_embeddings=intent_embeddings.get_all_embeddings(),
    hierarchy=hierarchy,
    classifier=classifier,
    custom_thresholds=None,  # No thresholds for belief extraction
    enable_belief_tracking=False  # Disable tracking for efficiency
)

# Extract beliefs
print("\nExtracting belief values (no clarifications)...")
results = extract_beliefs_from_data(
    data_loader=data_loader,
    ds_calculator=ds_calculator,
    dataset_name=DATASET,
    split='test',
    num_samples=EVAL_SAMPLES,
    n_chunks=4,
    output_dir=RESULTS_DIR
)

# Save results
print("\nSaving results...")
beliefs_df = pd.DataFrame(results)
beliefs_file = RESULTS_DIR / f"{DATASET}_beliefs.csv"
beliefs_df.to_csv(beliefs_file, index=False)

print(f"\n✅ STEP 1 Complete!")
print(f"   Total samples: {len(beliefs_df)}")
print(f"   Number of intents: {len(beliefs_df.columns) - 2}")
print(f"   Saved to: {beliefs_file}")


2026-02-05 19:34:22,199 - INFO - Loaded hierarchy from /home/kudzai/projects/DS_Project/config/hierarchies/banking77_hierarchy.json
2026-02-05 19:34:22,199 - INFO - Loaded hierarchical intents from /home/kudzai/projects/DS_Project/config/hierarchies/banking77_intents.json


STEP 1: BELIEF EXTRACTION

Loading hierarchy and intents...
Creating intent embeddings...


2026-02-05 19:34:22,485 - INFO - Computed embeddings for 96 intents
2026-02-05 19:34:22,486 - INFO - Use pytorch device_name: cuda
2026-02-05 19:34:22,487 - INFO - Load pretrained SentenceTransformer: intfloat/e5-base


Initializing DS Mass Function...


2026-02-05 19:34:24,000 - INFO - Loaded embedding model: intfloat/e5-base
2026-02-05 19:34:24,000 - INFO - Loading test data...
2026-02-05 19:34:24,000 - INFO - Processing all 3080 samples
2026-02-05 19:34:24,044 - INFO - 
Processing chunk 1/4 (770 samples)



Extracting belief values (no clarifications)...


Chunk 1: 100%|██████████| 770/770 [00:03<00:00, 213.98it/s]
2026-02-05 19:34:27,723 - INFO - Saved chunk 1 to /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/banking77_beliefs_chunk_1.csv
2026-02-05 19:34:27,723 - INFO - 
Processing chunk 2/4 (770 samples)
Chunk 2: 100%|██████████| 770/770 [00:03<00:00, 214.22it/s]
2026-02-05 19:34:31,398 - INFO - Saved chunk 2 to /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/banking77_beliefs_chunk_2.csv
2026-02-05 19:34:31,399 - INFO - 
Processing chunk 3/4 (770 samples)
Chunk 3: 100%|██████████| 770/770 [00:03<00:00, 213.93it/s]
2026-02-05 19:34:35,079 - INFO - Saved chunk 3 to /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/banking77_beliefs_chunk_3.csv
2026-02-05 19:34:35,080 - INFO - 
Processing chunk 4/4 (770 samples)
Chunk 4: 100%|██████████| 770/770 [00:03<00:00, 213.32it/s]
2026-02-05 19:34:38,769 - INFO - Saved chunk 4 to /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/ban


Saving results...

✅ STEP 1 Complete!
   Total samples: 3080
   Number of intents: 96
   Saved to: /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/banking77_beliefs.csv


## STEP 2: Threshold Computation

Analyze belief distributions and compute optimal thresholds per intent.

**What it does:**
- Analyze beliefs.csv
- Find threshold that maximizes F1 for each intent
- Balance precision and recall

**Output:** `optimal_thresholds.json`

In [4]:
# STEP 2: Threshold Computation
from scripts.ds_preparation.compute_thresholds import compute_optimal_thresholds, add_ancestor_labels
import json

print("="*60)
print("STEP 2: THRESHOLD COMPUTATION")
print("="*60)

# Load beliefs DataFrame (already loaded from STEP 1)
print(f"\nLoaded beliefs with {len(beliefs_df)} samples")

# Add ancestor-aware ground truth labels
print("Adding ancestor-aware ground truth labels...")
beliefs_df_with_labels = add_ancestor_labels(beliefs_df, hierarchy)

# Identify intent columns (those without 'is_correct_' prefix)
intent_columns = [
    col for col in beliefs_df_with_labels.columns
    if not col.startswith('is_correct_') and col not in ['query', 'true_intent']
]
print(f"Found {len(intent_columns)} intent columns")

# Compute optimal thresholds
print("Computing optimal thresholds...")
optimal_results = compute_optimal_thresholds(
    df=beliefs_df_with_labels,
    intent_columns=intent_columns
)

# Save results
thresholds_file = RESULTS_DIR / f"{DATASET}_optimal_thresholds.json"
with open(thresholds_file, 'w') as f:
    json.dump(optimal_results, f, indent=4)

print(f"\n✅ STEP 2 Complete!")
print(f"   Number of intents: {len(optimal_results)}")
print(f"   Saved to: {thresholds_file}")


STEP 2: THRESHOLD COMPUTATION

Loaded beliefs with 3080 samples
Adding ancestor-aware ground truth labels...


2026-02-05 19:34:51,238 - INFO - Computing optimal thresholds for 96 intents...


Found 96 intent columns
Computing optimal thresholds...


Processing intents: 100%|██████████| 96/96 [00:05<00:00, 16.08it/s]


✅ STEP 2 Complete!
   Number of intents: 96
   Saved to: /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/banking77_optimal_thresholds.json


## STEP 3: DS Evaluation WITH Optimal Thresholds

Evaluate DS system with optimal thresholds (from STEP 2).

**What it does:**
- Run DS with custom_thresholds = optimal_thresholds.json
- LLM-based user simulation for clarifications
- Measure: Accuracy, F1, avg_clarifications

**Output:** `ds_predictions.csv` with full interaction logs


In [5]:
from src.agents.customer_agent import CustomerAgent
from config.threshold_loader import load_thresholds_from_json
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import json

print("="*60)
print("STEP 3: DS EVALUATION WITH OPTIMAL THRESHOLDS")
print("="*60)

# Initialize LLM-based customer agent
print("\nInitializing LLM-based customer agent...")
customer_agent = CustomerAgent(rate_limit_delay=0.1)  # 0.1s for faster testing

# Get test data
test_texts, test_labels, _ = data_loader.get_split_data('test')

# Show sample data
sample_data = [{'text': test_texts[i], 'label': test_labels[i]} for i in range(min(10, len(test_texts)))]
print(f"\nSample of test data (first 10 samples):")
print(sample_data)

# Load optimal thresholds using the utility function
print(f"\nLoading optimal thresholds from: {thresholds_file}")
optimal_thresholds = load_thresholds_from_json(str(thresholds_file))
print(f"Loaded {len(optimal_thresholds)} custom thresholds")
print(f"Sample thresholds: {list(optimal_thresholds.items())[:5]}")

# Create ds_calculator WITH optimal thresholds
ds_calculator = DSMassFunction(
    intent_embeddings=intent_embeddings.get_all_embeddings(),
    hierarchy=hierarchy,
    classifier=classifier,
    custom_thresholds=optimal_thresholds,  # Use optimal thresholds from STEP 2 (as dict!)
    enable_belief_tracking=False
)
ds_calculator.customer_agent_callback = customer_agent

# Evaluate with LLM simulation
print("\nEvaluating DS with optimal thresholds (LLM-based user simulation)...")
results = []
interaction_count = 0

for idx, (text, true_intent) in enumerate(zip(test_texts, test_labels)):
    # Reset conversation
    ds_calculator.conversation_history = []
    ds_calculator.clear_belief_history()
    
    # Compute initial mass and get prediction
    initial_mass = ds_calculator.compute_mass_function(text)
    DS_prediction = ds_calculator.evaluate_with_clarifications(initial_mass)
    
    # Extract prediction
    if DS_prediction:
        pred_intent, confidence = DS_prediction
    else:
        pred_intent, confidence = "unknown", 0.0
    
    # Store result
    result = {
        'query': text,
        'true_intent': true_intent,
        'predicted_intent': pred_intent,
        'confidence': confidence,
        'interaction': '\n'.join(ds_calculator.conversation_history)
    }
    results.append(result)
    
    # Print conversation
    if ds_calculator.conversation_history:
        interaction_count += 1
        print(f"\n--- Interaction {interaction_count} ---")
        print(f"Query: {text}")
        print(f"True intent: {true_intent}")
        print(f"Predicted: {pred_intent}")
        for turn in ds_calculator.conversation_history:
            print(turn)
    
    # Progress indicator
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_texts)} samples...")

# Save results to CSV
output_dir = RESULTS_DIR / "ds_evaluation"
output_dir.mkdir(parents=True, exist_ok=True)
results_df = pd.DataFrame(results)
results_file = output_dir / f"{DATASET}_predictions.csv"
results_df.to_csv(results_file, index=False)

print(f"\n{'='*60}")
print(f"✅ EVALUATION COMPLETE!")
print(f"   Total samples: {len(results)}")
print(f"   Samples with clarifications: {interaction_count}")
print(f"   Saved to: {results_file}")
print(f"{'='*60}")

# ============================================================================
# ANALYSIS 
# ============================================================================

print("\n" + "="*60)
print("ANALYZING RESULTS")
print("="*60)

# Load predictions and add analysis columns
df = pd.DataFrame(results)
df['num_clarifications'] = df['interaction'].fillna('').str.count('Chatbot:')
df['is_correct'] = df['true_intent'] == df['predicted_intent']
df['has_clarifications'] = df['num_clarifications'] > 0

# Calculate metrics
print("\n" + "="*60)
print("OVERALL METRICS")
print("="*60)
accuracy = accuracy_score(df['true_intent'], df['predicted_intent'])
f1 = f1_score(df['true_intent'], df['predicted_intent'], average='macro', zero_division=0)
avg_clarifications = df['num_clarifications'].mean()
samples_with_clarif = df['has_clarifications'].sum()
clarification_rate = (samples_with_clarif / len(df)) * 100

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 (macro): {f1:.4f}")
print(f"Avg clarifications per query: {avg_clarifications:.2f}")
print(f"Queries with clarifications: {samples_with_clarif}/{len(df)} ({clarification_rate:.1f}%)")

# Analyze clarification effectiveness
print("\n" + "="*60)
print("CLARIFICATION EFFECTIVENESS")
print("="*60)

clarif_df = df[df['has_clarifications']]
if len(clarif_df) > 0:
    clarif_accuracy = (clarif_df['is_correct']).mean()
    print(f"Accuracy on queries with clarifications: {clarif_accuracy:.4f}")
    print(f"Total clarification turns: {clarif_df['num_clarifications'].sum()}")
    print(f"Avg turns per clarified query: {clarif_df['num_clarifications'].mean():.2f}")
else:
    print("No clarifications needed!")

# Save comprehensive analysis
analysis_file = output_dir / "analysis.json"
analysis_data = {
    'accuracy': float(accuracy),
    'f1_macro': float(f1),
    'avg_clarifications': float(avg_clarifications),
    'samples_with_clarifications': int(samples_with_clarif),
    'clarification_rate_percent': float(clarification_rate),
    'total_samples': len(df),
    'total_clarification_turns': int(clarif_df['num_clarifications'].sum() if len(clarif_df) > 0 else 0),
    'accuracy_on_clarified_queries': float((clarif_df['is_correct']).mean()) if len(clarif_df) > 0 else None
}

with open(analysis_file, 'w') as f:
    json.dump(analysis_data, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ ANALYSIS COMPLETE!")
print(f"   Full results saved to: {results_file}")
print(f"   Analysis saved to: {analysis_file}")
print(f"{'='*60}")

2026-02-05 19:35:04,327 - INFO - Initialized CustomerAgent with model: gpt-4o-mini
2026-02-05 19:35:04,343 - INFO - Loaded thresholds for 96 intents
2026-02-05 19:35:04,344 - INFO - Use pytorch device_name: cuda
2026-02-05 19:35:04,344 - INFO - Load pretrained SentenceTransformer: intfloat/e5-base


STEP 3: DS EVALUATION WITH OPTIMAL THRESHOLDS

Initializing LLM-based customer agent...

Sample of test data (first 10 samples):
[{'text': 'How do I locate my card?', 'label': 'card_arrival'}, {'text': 'I still have not received my new card, I ordered over a week ago.', 'label': 'card_arrival'}, {'text': 'I ordered a card but it has not arrived. Help please!', 'label': 'card_arrival'}, {'text': 'Is there a way to know when my card will arrive?', 'label': 'card_arrival'}, {'text': 'My card has not arrived yet.', 'label': 'card_arrival'}, {'text': 'When will I get my card?', 'label': 'card_arrival'}, {'text': 'Do you know if there is a tracking number for the new card you sent me?', 'label': 'card_arrival'}, {'text': 'i have not received my card', 'label': 'card_arrival'}, {'text': 'still waiting on that card', 'label': 'card_arrival'}, {'text': 'Is it normal to have to wait over a week for my new card?', 'label': 'card_arrival'}]

Loading optimal thresholds from: /home/kudzai/projects/D

2026-02-05 19:35:05,868 - INFO - Loaded embedding model: intfloat/e5-base



Evaluating DS with optimal thresholds (LLM-based user simulation)...


2026-02-05 19:35:07,805 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:35:09,639 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:35:14,292 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:35:16,421 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:35:19,139 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:35:19,164 - INFO - Single confident leaf node found: card_arrival
2026-02-05 19:35:19,169 - INFO - Single confident leaf node found: card_arrival
2026-02-05 19:35:19,175 - INFO - Multiple confident leaf nodes found. LCA: Physical Cards


...


--- Interaction 3025 ---
Query: do the details of my profile have to match my documents
True intent: verify_my_identity
Predicted: why_verify_identity
User: do the details of my profile have to match my documents
Chatbot: I'm not entirely sure what you're asking. Could you rephrase your question a bit?
User: I think there might be a misunderstanding. I was asking if the information in my profile needs to match the documents I have, like my ID or other official papers. Can you help me with that?

--- Interaction 3026 ---
Query: Where do I verify my identity?
True intent: verify_my_identity
Predicted: verify_my_identity
User: Where do I verify my identity?

--- Interaction 3027 ---
Query: When getting my ID checked, what are the steps involved?
True intent: verify_my_identity
Predicted: verify_my_identity
User: When getting my ID checked, what are the steps involved?


2026-02-05 19:56:57,283 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:56:57,300 - INFO - Multiple confident leaf nodes found. LCA: Verification
2026-02-05 19:56:58,203 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:56:58,227 - INFO - Multiple confident leaf nodes found. LCA: Verification
2026-02-05 19:57:00,186 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:00,209 - INFO - Multiple confident leaf nodes found. LCA: Verification
2026-02-05 19:57:03,806 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:03,827 - INFO - Multiple confident leaf nodes found. LCA: Verification
2026-02-05 19:57:05,164 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:05,194 - INFO - Single confident leaf node found: verify_my_identi


--- Interaction 3028 ---
Query: do I need anything for the identity check?
True intent: verify_my_identity
Predicted: unknown
User: do I need anything for the identity check?
Chatbot: It seems like you're looking for something related to Verification. Could you clarify which specific thing you're interested in? Here are a few suggestions: (['verify_my_identity', 'why_verify_identity', 'unable_to_verify_identity'])
User: I'm just trying to figure out what I need for the identity check. Do I need to provide any specific documents or information?
Chatbot: It seems like you're looking for something related to Verification. Could you clarify which specific thing you're interested in? Here are a few suggestions: (['why_verify_identity', 'verify_my_identity', 'unable_to_verify_identity'])
User: I just want to know what documents or info I need to provide for the identity check. Can you help me with that?
Chatbot: It seems like you're looking for something related to Verification. Could you c

2026-02-05 19:57:06,426 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:07,903 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:09,167 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:10,373 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:11,418 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-05 19:57:11,442 - INFO - Single confident leaf node found: country_support
2026-02-05 19:57:11,447 - INFO - Single confident leaf node found: country_support
2026-02-05 19:57:11,452 - INFO - Single confident leaf node found: country_support
2026-02-05 19:57:11,457 - INFO - Single confident leaf node found: country_support
2026-02-05 19:57:11,462 - INFO - Single confident leaf node found: country_support
2026-02-05 19:57:11,


--- Interaction 3059 ---
Query: Where can I find your locations?
True intent: country_support
Predicted: unknown
User: Where can I find your locations?
Chatbot: I'm not entirely sure what you're asking. Could you rephrase your question a bit?
User: I think there might be a misunderstanding. I was asking where I can find your locations, like your stores or offices. Can you help me with that?
Chatbot: There are a few things that might match: (['country_support', 'edit_personal_details', 'verify_source_of_funds']). Could you clarify a bit more?
User: Hmm, those options don't really seem to relate to what I was asking about. I just want to know where I can find your physical stores or offices. Can you help me with that?
Chatbot: There are a few things that might match: (['country_support', 'edit_personal_details', 'verify_source_of_funds']). Could you clarify a bit more?
User: I appreciate you trying to help, but those options still don't really relate to what I'm asking. I'm just looking

In [9]:
# DEBUG: Check if interactions were actually saved
import pandas as pd

# Define the output directory from STEP 3
predictions_output_dir = RESULTS_DIR / "ds_evaluation"
predictions_file = predictions_output_dir / f"{DATASET}_predictions.csv"

if predictions_file.exists():
    df = pd.read_csv(predictions_file)
    print(f"Loaded predictions CSV: {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nSample rows with interaction column:")
    
    # Show non-empty interactions
    has_interactions = df[df['interaction'].notna() & (df['interaction'].str.len() > 0)]
    print(f"Rows with interactions: {len(has_interactions)}")
    
    if len(has_interactions) > 0:
        print(f"\nFirst interaction:")
        print(has_interactions.iloc[0]['interaction'])
    else:
        print("\nNo interactions found in CSV!")
        print(f"\nFirst 3 rows:")
        print(df.head(3))
else:
    print(f"❌ Predictions file not found: {predictions_file}")
    print("Make sure to run STEP 3 (cell 11) first!")


Loaded predictions CSV: 3080 rows
Columns: ['query', 'true_intent', 'predicted_intent', 'confidence', 'interaction']

Sample rows with interaction column:
Rows with interactions: 3080

First interaction:
User: How do I locate my card?
Chatbot: I'm not entirely sure what you're asking. Could you rephrase your question a bit?
User: I think there might be some confusion here. I was asking how to locate my card, like a debit or credit card. Can you help me with that?
Chatbot: There are a few things that might match: (['card_linking', 'lost_or_stolen_card']). Could you clarify a bit more?
User: Sure! I'm trying to figure out if I lost my card or if I just need help linking it to my account. I think I might have misplaced it, so I guess I'm looking for help with a lost or stolen card. What should I do next?
Chatbot: There are a few things that might match: (['card_linking', 'lost_or_stolen_card']). Could you clarify a bit more?
User: I already mentioned that I'm looking for help with a lost 

## STEP 5: Select User Study Queries

Select samples from STEP 4 results for actual human user study.

**What it does:**
- Uses STEP 4 results (DS with optimal thresholds)
- Prioritizes queries that needed clarifications
- Selects based on: low confidence, misclassifications, multiple turns
- Output top N queries for human validation

**Output:** `selected_queries_for_user_study.csv`

In [10]:
import sys
sys.path.insert(0, str(BASE_DIR))
from src.utils.query_selector import QuerySelector
import pandas as pd

print("="*60)
print("STEP 5: SELECT USER STUDY QUERIES")
print("="*60)

# Load predictions from STEP 3
predictions_file = RESULTS_DIR / "ds_evaluation" / f"{DATASET}_predictions.csv"
print(f"\nLoading predictions from: {predictions_file}")

if not predictions_file.exists():
    print(f"❌ Predictions file not found!")
    print("Please run STEP 3 (cell 11) first to generate predictions.")
else:
    # Load results
    results_df = pd.read_csv(predictions_file)
    print(f"Loaded {len(results_df)} predictions")
    
    # Initialize QuerySelector
    selector = QuerySelector(
        min_interactions=2,      # Minimum 2 clarification turns
        max_confidence=0.7,      # Low confidence threshold
        include_incorrect=True   # Include misclassifications
    )
    
    # Select queries for user study
    print(f"\nSelecting {HUMAN_STUDY_SAMPLES} queries using 'balanced' strategy...")
    selected_queries = selector.select_for_user_study(
        results_df=results_df,
        max_samples=HUMAN_STUDY_SAMPLES,
        strategy='balanced'  # Mix of all problem categories
    )
    
    # Generate summary
    summary = selector.generate_study_summary(selected_queries)
    
    # Save selected queries
    selected_file = USER_STUDY_DIR / "selected_queries_for_user_study.csv"
    selected_file.parent.mkdir(parents=True, exist_ok=True)
    selected_queries.to_csv(selected_file, index=False)
    
    print(f"\n✅ STEP 5 Complete!")
    print(f"   Selected {len(selected_queries)} queries")
    print(f"   Avg clarifications: {summary['avg_interactions']:.2f}")
    print(f"   Avg confidence: {summary['avg_confidence']:.4f}")
    print(f"   LLM accuracy on selected: {summary['accuracy']:.4f}")
    print(f"\n   Category distribution:")
    for cat, count in summary['categories'].items():
        print(f"      {cat}: {count}")
    print(f"   Saved to: {selected_file}")


2026-02-05 20:38:29,520 - INFO - Found 205 queries with >= 2 interactions
2026-02-05 20:38:29,521 - INFO - Found 2741 queries with confidence <= 0.7
2026-02-05 20:38:29,521 - INFO - Found 330 incorrect predictions
2026-02-05 20:38:29,522 - INFO - Found 142 queries that are both high-interaction and incorrect
2026-02-05 20:38:29,522 - INFO - Found 272 queries that are both low-confidence and incorrect
2026-02-05 20:38:29,524 - INFO - Selected 91 queries for user study


STEP 5: SELECT USER STUDY QUERIES

Loading predictions from: /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/ds_evaluation/banking77_predictions.csv
Loaded 3080 predictions

Selecting 100 queries using 'balanced' strategy...

✅ STEP 5 Complete!
   Selected 91 queries
   Avg clarifications: 2.07
   Avg confidence: 0.3897
   LLM accuracy on selected: 0.2418

   Category distribution:
      high_interaction: 20
      low_confidence: 20
      incorrect: 19
      problematic: 18
      uncertain: 14
   Saved to: /home/kudzai/projects/DS_Project/outputs/user_study/workflow_demo/selected_queries_for_user_study.csv


## STEP 6: Real User Study (Streamlit Interface)

**Real human participants replace the LLM agent on challenging queries identified in STEP 5.**

### 🎯 Purpose

In STEP 3, the LLM simulated agent processed all 3,080 queries. STEP 5 identified **91 challenging queries** where the system struggled:

- **High interactions**: Queries needing multiple clarifications (avg 2.07 turns)
- **Low confidence**: Uncertain predictions (avg confidence 0.3897)
- **Low LLM accuracy**: Only 24.18% correct on these challenging cases
- **Categories**: High-interaction, low-confidence, incorrect, problematic, uncertain

**STEP 6 validates** how well the system performs with **real human users** on these edge cases.

### 🚀 Launch the Streamlit App

```bash
cd /home/kudzai/projects/DS_Project
streamlit run app_main.py
```

Then navigate to **"💬 User Study"** page in the sidebar.

### ✨ What Happens in the Interface

1. **Loads 91 selected queries** from STEP 5 automatically
2. **Presents queries one-by-one** to human participants
3. **Human provides clarifications** (replaces LLM CustomerAgent)
4. **Tracks belief progression** in real-time as humans respond
5. **Validates predictions** (correct/incorrect)
6. **Saves session data** for STEP 7 comparison analysis

### 📊 Key Differences: LLM vs Human

| Aspect | STEP 3 (LLM Agent) | STEP 6 (Real User - Streamlit) |
|--------|-------------------|--------------------------------|
| **Who responds** | GPT-4o-mini CustomerAgent | Real human participant |
| **Query set** | All 3,080 test queries | 91 challenging queries only |
| **Purpose** | Identify problematic queries | Validate system with humans |
| **Execution** | Automated (Jupyter notebook) | Interactive (Streamlit UI) |
| **Output** | `ds_evaluation/banking77_predictions.csv` | `human_study/session_*.csv` |

### 📁 Configuration (Auto-loaded in Streamlit)

The interface uses files generated from previous steps:

- **Queries**: `outputs/user_study/workflow_demo/selected_queries_for_user_study.csv` (STEP 5)
- **Model**: `experiments/banking77/banking77_logistic_model.pkl` (STEP 0)
- **Thresholds**: `results/banking77/workflow_demo/banking77_optimal_thresholds.json` (STEP 2)
- **Hierarchy**: `config/hierarchies/banking77_hierarchy.json`
- **Intents**: `config/hierarchies/banking77_intents.json`

### 💾 Output Files

```
outputs/user_study/human_study/
├── session_20260205_143015.csv    # User 1's session
├── session_20260205_153042.csv    # User 2's session
└── progress_20260205_144522.csv   # Mid-session checkpoint
```

Each CSV contains:
- Query text and ground truth
- Predicted intent and confidence
- Number of clarification turns
- Full conversation transcript
- Correctness validation
- Timestamp

### 🔄 Workflow Position

```
STEP 3: LLM Simulation (3,080 queries)
    ↓
    Identifies challenging queries
    ↓
STEP 5: Query Selection (91 queries)
    ↓
    High interaction, low confidence, incorrect predictions
    ↓
STEP 6: Real User Study (Streamlit) ← You are here
    ↓
    Human validation on challenging queries
    ↓
STEP 7: Comparison Analysis
    Vanilla baseline vs LLM vs Real users
```

### 📋 Quick Start Steps

1. **Verify prerequisites**: Ensure STEP 0-5 are complete
2. **Launch Streamlit**: `streamlit run app_main.py`
3. **Navigate**: Click "💬 User Study" in sidebar
4. **Load config**: Click "📥 Load Configuration" (uses defaults)
5. **Process queries**: Read → Clarify → Validate → Next
6. **Download results**: Click "📥 Download Session Data" when done

### 📚 Documentation

- **Detailed Guide**: [STEP6_USER_STUDY_GUIDE.md](../STEP6_USER_STUDY_GUIDE.md)
- **Streamlit App README**: [src/streamlit_app/README.md](../src/streamlit_app/README.md)

### 🌐 Deployment

This interface can be deployed to **Streamlit Cloud** for public access:

```bash
# Push to GitHub
git add .
git commit -m "Add STEP 6 user study interface"
git push origin main

# Deploy on Streamlit Cloud
# Visit: https://share.streamlit.io
# Connect repo and deploy app_main.py
```

**Note:** This is the canonical implementation for real user studies. For reproducibility, share your Streamlit Cloud link with participants.

## STEP 7: Comparison Report

Compare all evaluation methods and show improvements.

**What it compares:**
- Vanilla Baseline (LogisticRegression + BERT)
- DS without thresholds (baseline DS)
- DS with optimal thresholds (optimized DS)
- Real user study results (if available)

**Output:** Comparison table, plots, improvement percentages

In [ ]:
from scripts.analysis.compare_results import compare_all_results
import pandas as pd

print("="*60)
print("STEP 7: COMPARISON REPORT")
print("="*60)

# Prepare comparison
comparison_dir = RESULTS_DIR / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)

# Compare all methods
print("\nComparing all evaluation methods...")
comparison_df = compare_all_results(
    vanilla_metrics_file=vanilla_output_dir / f"{DATASET}_vanilla_metrics.json",
    ds_no_thresh_metrics_file=no_thresh_output_dir / "metrics.json",
    ds_with_thresh_metrics_file=with_thresh_output_dir / "metrics.json",
    output_dir=comparison_dir
)

print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

print(f"\n✅ STEP 7 Complete!")
print(f"   Detailed results saved to: {comparison_dir}")
print(f"   • comparison_table.csv")
print(f"   • comparison_plots.png")
print(f"   • comparison_summary.txt")

In [ ]:
# Visualize selection results
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of clarifications
axes[0, 0].hist(selected_queries['num_clarifications'], bins=range(0, 6), edgecolor='black')
axes[0, 0].set_xlabel('Number of Clarifications')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Clarification Distribution (Selected Queries)')

# 2. Initial vs Final accuracy
initial_acc = (selected_queries['initial_predicted_intent'] == selected_queries['true_intent']).mean()
final_acc = (selected_queries['final_predicted_intent'] == selected_queries['true_intent']).mean()
axes[0, 1].bar(['Initial', 'Final'], [initial_acc, final_acc], color=['red', 'green'])
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy Before/After Clarifications')
axes[0, 1].set_ylim([0, 1])

# 3. Confidence distribution
axes[1, 0].hist(selected_queries['initial_confidence'], bins=20, alpha=0.7, label='Initial', edgecolor='black')
axes[1, 0].hist(selected_queries['final_confidence'], bins=20, alpha=0.7, label='Final', edgecolor='black')
axes[1, 0].set_xlabel('Confidence Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Confidence Distribution')
axes[1, 0].legend()

# 4. Selection priority distribution
axes[1, 1].hist(selected_queries['selection_priority'], bins=15, edgecolor='black')
axes[1, 1].set_xlabel('Selection Priority Score')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Query Selection Priority')

plt.tight_layout()
plt.show()

## Phase 4: Human User Study (Manual)

The selected queries are now ready for human user study. To run the study:

```bash
python scripts/user_study/run_user_study.py --study-queries outputs/user_study/workflow_demo/selected_queries_for_user_study.csv
```

This will launch the interactive study interface for human participants.

## Phase 3.5: Explainability Analysis

Analyze belief progression and validate prediction faithfulness.

**Purpose:** Understand WHY the system makes decisions and ensure explanations are trustworthy.

**Output:**
- Belief progression statistics
- Faithfulness validation metrics
- Visualization: `explainability_analysis.png`

In [ ]:
import subprocess
import sys

print("="*60)
print("PHASE 3.5: EXPLAINABILITY ANALYSIS")
print("="*60)

# Configuration
belief_logs_dir = RESULTS_DIR / "with_thresholds" / "belief_logs"
predictions_file = RESULTS_DIR / "with_thresholds" / f"{DATASET}_predictions.csv"
explainability_dir = RESULTS_DIR / "explainability"

# Build explainability analysis command
explain_script = BASE_DIR / "scripts" / "analysis" / "analyze_explainability.py"
cmd = [
    sys.executable, str(explain_script),
    "--belief-logs-dir", str(belief_logs_dir),
    "--predictions-file", str(predictions_file),
    "--output-dir", str(explainability_dir),
    "--max-samples", "100"
]

# Run explainability analysis
print("\nAnalyzing belief progression and faithfulness...")
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError("Explainability analysis failed")

# Load results
import json
explainability_metrics_file = explainability_dir / "explainability_metrics.json"
with open(explainability_metrics_file) as f:
    explainability_metrics = json.load(f)

print("\n✅ Explainability Analysis Complete!")
if 'belief_progression' in explainability_metrics:
    bp = explainability_metrics['belief_progression']
    print(f"   Queries analyzed: {bp.get('analyzed_queries', 0)}")
    print(f"   Avg turns: {bp.get('avg_turns', 0):.2f}")
    print(f"   Avg belief convergence: {bp.get('avg_belief_convergence', 0):.3f}")

if 'faithfulness' in explainability_metrics:
    faith = explainability_metrics['faithfulness']
    print(f"   Faithfulness rate: {faith.get('faithfulness_rate', 0):.2%}")
    print(f"   Faithful predictions: {faith.get('faithful_predictions', 0)}/{faith.get('total_validated', 0)}")

print(f"\n   Results saved to: {explainability_dir}")

## Summary: Compare Baseline vs Optimized DS

In [ ]:
# Comparison metrics
print("\n" + "="*60)
print("COMPLETE WORKFLOW SUMMARY")
print("="*60)

print("\n📊 Phase 1: Threshold Optimization")
print(f"   Baseline accuracy: {accuracy:.4f}")
print(f"   Optimized accuracy: {with_thresh_accuracy:.4f}")
print(f"   Improvement: {improvement:+.2f}%")

print("\n🤖 Phase 2: LLM Full Test Set Evaluation")
print(f"   Queries processed: {len(llm_df)}")
print(f"   Average clarifications: {llm_df['num_clarifications'].mean():.2f}")
llm_accuracy = (llm_df['final_predicted_intent'] == llm_df['true_intent']).mean()
print(f"   Final accuracy: {llm_accuracy:.4f}")
print(f"   Queries needing clarification: {(llm_df['num_clarifications'] > 0).sum()} ({(llm_df['num_clarifications'] > 0).mean()*100:.1f}%)")

print("\n👥 Phase 3: Human Study Query Selection")
print(f"   Queries selected: {len(selected_queries)}")
print(f"   Average priority score: {selected_queries['selection_priority'].mean():.2f}")
print(f"   Ready for human study: {selected_file}")

print("\n" + "="*60)

In [ ]:
# Final comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Accuracy comparison
stages = ['Baseline\n(No Thresholds)', 'Optimized\n(With Thresholds)', 'LLM Simulation\n(Full Test)']
accuracies = [
    accuracy,
    with_thresh_accuracy,
    llm_accuracy
]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
axes[0].bar(stages, accuracies, color=colors)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim([0.7, 1.0])
axes[0].set_title('System Accuracy Across Phases', fontsize=14, fontweight='bold')
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# 2. Clarification effectiveness
clarification_data = llm_df[llm_df['num_clarifications'] > 0]
if len(clarification_data) > 0:
    initial = (clarification_data['initial_predicted_intent'] == clarification_data['true_intent']).mean()
    final = (clarification_data['final_predicted_intent'] == clarification_data['true_intent']).mean()
    axes[1].bar(['Before\nClarification', 'After\nClarification'], [initial, final], color=['#ff6b6b', '#51cf66'])
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_ylim([0, 1])
    axes[1].set_title('Clarification Impact', fontsize=14, fontweight='bold')
    for i, v in enumerate([initial, final]):
        axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# 3. Query selection priority distribution
priority_bins = selected_queries['selection_priority'].value_counts().sort_index()
axes[2].bar(priority_bins.index, priority_bins.values, color='#9775fa')
axes[2].set_xlabel('Priority Score', fontsize=12)
axes[2].set_ylabel('Number of Queries', fontsize=12)
axes[2].set_title(f'Human Study Query Selection\n(Top {HUMAN_STUDY_SAMPLES} Queries)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 🎯 Next Steps

### Research Validation Complete
This notebook demonstrates the complete DS-based intent clarification system workflow:

1. ✅ **Phase 1 (Configure)**: Trained model, computed optimal thresholds, validated improvement
2. ✅ **Phase 2 (LLM Evaluation)**: Simulated user interactions across full test set
3. ✅ **Phase 3 (Selection)**: Identified most problematic queries for human validation
4. ⏳ **Phase 4 (Human Study)**: Ready to run with real users

### Key Findings from This Run
The workflow successfully demonstrates:
- **Threshold Optimization**: Improved classification accuracy through belief-based thresholds
- **Clarification Effectiveness**: DS agent resolves ambiguous queries through interactive clarification
- **System Readiness**: Challenging queries selected for human validation study

### Running Human Study
To conduct the human user study with selected queries:

```bash
# Activate environment
conda activate dsproject

# Run user study
python scripts/user_study/run_user_study.py \
    --study-queries outputs/user_study/workflow_demo/selected_queries_for_user_study.csv \
    --model experiments/banking77/banking77_logistic_model.pkl \
    --hierarchy-file config/banking77_config.json \
    --intents-file config/banking77_intents.json \
    --thresholds config/banking77_thresholds.json \
    --output-dir outputs/user_study/human_study
```

### Configuration Options

**For Full Evaluation** (not demo):
- Set `QUICK_DEMO = False` in cell 2
- This will:
  - Train on full training set (10,003 examples)
  - Evaluate on full test set (3,080 examples)
  - Run LLM simulation on all 3,080 queries
  - Select 100 most challenging queries for human study

**For Quick Testing** (current):
- Keep `QUICK_DEMO = True`
- Uses 50 evaluation samples and 100 LLM samples
- Faster iteration for development and debugging

### Output Files Generated
- `experiments/banking77/banking77_logistic_model.pkl` - Trained classifier
- `results/banking77/workflow_demo/banking77_beliefs.csv` - Belief distributions
- `results/banking77/workflow_demo/banking77_thresholds.json` - Optimal thresholds
- `results/banking77/workflow_demo/with_thresholds/banking77_predictions.csv` - Final predictions
- `outputs/user_study/workflow_demo/llm_full_test/session_llm_full_eval.csv` - LLM results
- `outputs/user_study/workflow_demo/selected_queries_for_user_study.csv` - Human study queries

### Citation
If you use this system in your research, please cite our work on Dempster-Shafer based intent clarification with hierarchical reasoning.

## Phase 5: Comprehensive Comparison

Compare ALL evaluation methods with explainability metrics.

**Compares:**
- Vanilla Baseline (LogisticRegression + BERT)
- DS without thresholds
- DS with optimal thresholds
- LLM simulation results
- Explainability metrics

**Output:**
- Comparison table and plots
- Summary report with key findings
- Faithfulness comparison

In [ ]:
import subprocess
import sys

print("="*60)
print("PHASE 5: COMPREHENSIVE COMPARISON")
print("="*60)

# Configuration
vanilla_metrics = RESULTS_DIR / "vanilla" / f"{DATASET}_vanilla_metrics.json"
ds_with_thresh_metrics = RESULTS_DIR / "with_thresholds" / f"{DATASET}_metrics.json" 
llm_simulation_file = USER_STUDY_DIR / "llm_simulation_results.csv"
comparison_dir = RESULTS_DIR / "comparison"

# Build comparison command
compare_script = BASE_DIR / "scripts" / "analysis" / "compare_results.py"
cmd = [
    sys.executable, str(compare_script),
    "--vanilla-metrics", str(vanilla_metrics),
    "--ds-with-thresh-metrics", str(ds_with_thresh_metrics),
    "--output-dir", str(comparison_dir)
]

# Add optional LLM simulation results if available
if llm_simulation_file.exists():
    cmd.extend(["--llm-simulation-results", str(llm_simulation_file)])

# Run comparison
print("\nComparing all evaluation methods...")
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError("Comparison failed")

# Load and display comparison table
comparison_table = pd.read_csv(comparison_dir / "comparison_table.csv")

print("\n✅ Comparison Complete!")
print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
print(comparison_table.to_string(index=False))
print("="*80)

print(f"\n   Detailed results saved to: {comparison_dir}")
print(f"   • comparison_table.csv")
print(f"   • comparison_plots.png")
print(f"   • comparison_summary.txt")
print(f"   • all_results.json")